# 04 — Feature Engineering
**Goal**: Build every feature the model will train on. Features are grouped into four families:
1. **Raw activity features** — flight counts, distances, points (baseline)
2. **Behavioural economics features** — loss aversion, hyperbolic discounting, status quo bias
3. **Econometric features** — fixed-effects deviation, seasonal controls
4. **Demographic & tier features** — encoded categorical variables

**Critical rule**: Every feature is computed using ONLY data available at or before `PREDICTION_CUTOFF`. No future information crosses the boundary.

**Input**: `data/processed/02_activity_clean.csv` + `data/processed/02_cleaned.csv` + `data/processed/03_seasonal_volatility.csv`
**Output**: `data/processed/04_features.csv` — the single file notebook 05 trains on.

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

PROC = Path('../data/processed')
FIG  = Path('../reports/figures')
os.makedirs(FIG, exist_ok=True)

KEY          = 'loyalty_number'
YEAR_COL     = 'year'
MONTH_COL    = 'month'
FLIGHTS_COL  = 'total_flights'
DISTANCE_COL = 'distance'
POINTS_ACC   = 'points_accumulated'
POINTS_RED   = 'points_redeemed'
CLV_COL      = 'clv'
CARD_COL     = 'loyalty_card'

# Leakage boundary — ALL features must use data <= this date
PREDICTION_CUTOFF = pd.Timestamp('2016-12-01')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Feature registry
feature_registry = []
def register(name, family, description):
    feature_registry.append({'feature': name, 'family': family, 'description': description})

print('Setup complete. Prediction cutoff:', PREDICTION_CUTOFF.strftime('%Y-%m'))

---
## 1. Load Data

In [ ]:
members  = pd.read_csv(PROC / '02_cleaned.csv')
activity = pd.read_csv(PROC / '02_activity_clean.csv')
labels   = pd.read_csv(PROC / '02_churn_labels.csv')
sea_vol  = pd.read_csv(PROC / '03_seasonal_volatility.csv')[[KEY, 'seasonal_volatility']]

activity['period'] = pd.to_datetime(
    activity[YEAR_COL].astype(str) + '-' +
    activity[MONTH_COL].astype(str).str.zfill(2)
)

# Enforce leakage boundary
activity = activity[activity['period'] <= PREDICTION_CUTOFF].copy()

# Resolve actual column names
pts_acc_col = next((c for c in ['total_points_accumulated', POINTS_ACC] if c in activity.columns), POINTS_ACC)
pts_red_col = next((c for c in ['total_points_redeemed', POINTS_RED]    if c in activity.columns), POINTS_RED)
dist_col    = next((c for c in [DISTANCE_COL, 'distance_flown', 'km_flown'] if c in activity.columns), DISTANCE_COL)

# Active cohort only
cohort   = labels[KEY].tolist()
activity = activity[activity[KEY].isin(cohort)].copy()

print(f'Activity rows (post-cutoff): {len(activity):,}')
print(f'Members in cohort: {len(cohort):,}')
print(f'Columns — pts_acc: {pts_acc_col}, pts_red: {pts_red_col}, dist: {dist_col}')

---
## 2. Feature Family 1 — Raw Activity Baseline
Simple aggregates over full history. These are the baseline features — we will show behavioural features improve on them.

In [ ]:
agg_dict = {}
for col, fns in [(FLIGHTS_COL, ['sum','mean','max','std']),
                 (pts_acc_col, ['sum','mean']),
                 (pts_red_col, ['sum','mean']),
                 (dist_col,    ['sum','mean'])]:
    if col in activity.columns:
        agg_dict[col] = fns

raw_agg = activity.groupby(KEY).agg(agg_dict)
raw_agg.columns = ['_'.join(c).strip() for c in raw_agg.columns]
raw_agg = raw_agg.reset_index()

months_active = (
    activity[activity[FLIGHTS_COL] > 0]
    .groupby(KEY).size()
    .reset_index(name='months_with_flights')
)

tenure = activity.groupby(KEY)['period'].agg(first_month='min', last_month='max').reset_index()
tenure['tenure_months'] = (
    (tenure['last_month'].dt.year  - tenure['first_month'].dt.year)  * 12 +
    (tenure['last_month'].dt.month - tenure['first_month'].dt.month) + 1
)

features = (
    pd.DataFrame({KEY: cohort})
    .merge(raw_agg,       on=KEY, how='left')
    .merge(months_active, on=KEY, how='left')
    .merge(tenure[[KEY, 'tenure_months']], on=KEY, how='left')
)
features['months_with_flights'] = features['months_with_flights'].fillna(0)
features['activity_rate'] = (
    features['months_with_flights'] / features['tenure_months'].replace(0, np.nan)
).fillna(0)

for col in ['months_with_flights', 'tenure_months', 'activity_rate']:
    register(col, 'raw_activity', f'Historical {col}')
for col in raw_agg.columns[1:]:
    register(col, 'raw_activity', f'Aggregate: {col}')

print(f'Raw features: {features.shape[1]-1} features for {len(features):,} members')

---
## 3. Feature Family 2 — Behavioural Economics Features

### 3a. Hyperbolic Discounting — Recency-Weighted Engagement Score
**Theory**: Humans weight recent events disproportionately more than distant ones (Laibson, 1997). A member's recent flight behaviour is a stronger predictor than their lifetime total.

**Implementation**: Flights weighted by recency using a hyperbolic decay function.

In [ ]:
def recency_weight(months_ago, k=0.15):
    """Hyperbolic decay: w(t) = 1 / (1 + k*t)."""
    return 1.0 / (1.0 + k * months_ago)

act_12m = activity[
    activity['period'] >= PREDICTION_CUTOFF - pd.DateOffset(months=11)
].copy()

act_12m['months_ago'] = (
    (PREDICTION_CUTOFF.year  - act_12m['period'].dt.year)  * 12 +
    (PREDICTION_CUTOFF.month - act_12m['period'].dt.month)
)
act_12m['recency_weight']    = act_12m['months_ago'].apply(recency_weight)
act_12m['weighted_flights']  = act_12m[FLIGHTS_COL] * act_12m['recency_weight']

hyp_agg = {'weighted_flights': 'sum'}
if dist_col in act_12m.columns:
    act_12m['weighted_distance'] = act_12m[dist_col] * act_12m['recency_weight']
    hyp_agg['weighted_distance'] = 'sum'

hyperbolic = (
    act_12m.groupby(KEY).agg(hyp_agg).reset_index()
    .rename(columns={
        'weighted_flights' : 'hyperbolic_flight_score',
        'weighted_distance': 'hyperbolic_distance_score'
    })
)
features = features.merge(hyperbolic, on=KEY, how='left')
for col in [c for c in hyperbolic.columns if c != KEY]:
    features[col] = features[col].fillna(0)
    register(col, 'behavioural_hyperbolic',
             'Recency-weighted score — recent months weighted higher (hyperbolic discounting)')

# Rolling window totals
for window, label in [(3,'last_3m'), (6,'last_6m'), (12,'last_12m')]:
    wd = activity[activity['period'] > PREDICTION_CUTOFF - pd.DateOffset(months=window)]
    wa = (wd.groupby(KEY)[FLIGHTS_COL].sum().reset_index()
            .rename(columns={FLIGHTS_COL: f'flights_{label}'}))
    features = features.merge(wa, on=KEY, how='left')
    features[f'flights_{label}'] = features[f'flights_{label}'].fillna(0)
    register(f'flights_{label}', 'behavioural_recency', f'Total flights in last {window} months')

features['recency_ratio'] = (
    features['flights_last_3m'] / features['flights_last_12m'].replace(0, np.nan)
).fillna(0)
register('recency_ratio', 'behavioural_recency',
         'flights_last_3m / flights_last_12m — value < 0.25 signals declining trajectory')

print(f'After hyperbolic features: {features.shape[1]-1} features')

### 3b. Loss Aversion — Points Hoarding & Redemption Trend
**Theory**: People are more sensitive to potential losses than equivalent gains (Kahneman & Tversky, 1979). Members who accumulate but never redeem points are experiencing loss aversion — afraid to spend in case they need the points later.

In [ ]:
pts_totals = (
    activity.groupby(KEY)
    .agg({pts_acc_col: 'sum', pts_red_col: 'sum'})
    .reset_index()
    .rename(columns={pts_acc_col: 'total_pts_accumulated', pts_red_col: 'total_pts_redeemed'})
)
pts_totals['net_unredeemed_points'] = (
    pts_totals['total_pts_accumulated'] - pts_totals['total_pts_redeemed']
).clip(lower=0)
pts_totals['redemption_ratio'] = (
    pts_totals['total_pts_redeemed'] / pts_totals['total_pts_accumulated'].replace(0, np.nan)
).fillna(0).clip(0, 1)

pts_recent = (
    activity[activity['period'] > PREDICTION_CUTOFF - pd.DateOffset(months=6)]
    .groupby(KEY).agg({pts_acc_col: 'sum', pts_red_col: 'sum'})
    .reset_index()
    .rename(columns={pts_acc_col: 'pts_acc_last_6m', pts_red_col: 'pts_red_last_6m'})
)
pts_recent['recent_redemption_ratio'] = (
    pts_recent['pts_red_last_6m'] / pts_recent['pts_acc_last_6m'].replace(0, np.nan)
).fillna(0).clip(0, 1)

pts_all = pts_totals.merge(pts_recent, on=KEY, how='left').fillna(0)
pts_all['redemption_trend'] = pts_all['recent_redemption_ratio'] - pts_all['redemption_ratio']

# Loss aversion composite: high unredeemed + low redemption ratio
from sklearn.preprocessing import MinMaxScaler
scaler_tmp = MinMaxScaler()
pts_all['pts_hoard_norm'] = scaler_tmp.fit_transform(pts_all[['net_unredeemed_points']])
pts_all['loss_aversion_score'] = pts_all['pts_hoard_norm'] * (1 - pts_all['redemption_ratio'])

loss_cols = ['total_pts_accumulated','total_pts_redeemed','net_unredeemed_points',
             'redemption_ratio','recent_redemption_ratio','redemption_trend','loss_aversion_score']
features = features.merge(pts_all[[KEY] + loss_cols], on=KEY, how='left')
for col in loss_cols:
    features[col] = features[col].fillna(0)
    register(col, 'behavioural_loss_aversion', 'Points hoarding / redemption (loss aversion signal)')

print(f'After loss aversion features: {features.shape[1]-1} features')

### 3c. Status Quo Bias — Tier Position & Flight Trajectory
**Theory**: People tend to accept current state as default (Samuelson & Zeckhauser, 1988). Members with declining flights aren't loyal — they're inert.

In [ ]:
if CARD_COL in members.columns:
    tier_order = {
        'star':1,'nova':2,'aurora':3,
        'blue':1,'silver':2,'gold':3,'platinum':4,
        'bronze':1,'standard':1
    }
    member_tiers = members[[KEY, CARD_COL]].copy()
    member_tiers['tier_rank'] = (
        member_tiers[CARD_COL].str.lower().str.strip().map(tier_order)
    )
    if member_tiers['tier_rank'].isnull().any():
        from sklearn.preprocessing import LabelEncoder
        le_tier = LabelEncoder()
        member_tiers['tier_rank'] = le_tier.fit_transform(
            member_tiers[CARD_COL].fillna('unknown')
        )
        print('Tier encoding:', dict(zip(le_tier.classes_, le_tier.transform(le_tier.classes_))))

    first_6m = (
        activity.sort_values('period').groupby(KEY).head(6)
        .groupby(KEY)[FLIGHTS_COL].mean().reset_index()
        .rename(columns={FLIGHTS_COL: 'flights_first_6m_avg'})
    )
    last_6m_act = (
        activity[activity['period'] > PREDICTION_CUTOFF - pd.DateOffset(months=6)]
        .groupby(KEY)[FLIGHTS_COL].mean().reset_index()
        .rename(columns={FLIGHTS_COL: 'flights_last_6m_avg'})
    )
    traj_df = first_6m.merge(last_6m_act, on=KEY, how='outer').fillna(0)
    traj_df['flight_trajectory'] = traj_df['flights_last_6m_avg'] - traj_df['flights_first_6m_avg']

    stag_df = member_tiers.merge(traj_df, on=KEY, how='left').fillna(0)
    stag_cols = ['tier_rank','flight_trajectory','flights_first_6m_avg','flights_last_6m_avg']
    features = features.merge(stag_df[[KEY] + stag_cols], on=KEY, how='left')
    for col in stag_cols:
        features[col] = features[col].fillna(0)
        register(col, 'behavioural_status_quo', 'Tier rank and flight trajectory (status quo bias)')
    print(f'After status quo features: {features.shape[1]-1} features')
else:
    print(f'Card column not found — skipping tier features.')

### 3d. Engagement Consistency — Flight Regularity & Streaks

In [ ]:
cv_flights = (
    activity.groupby(KEY)[FLIGHTS_COL]
    .agg(mean_f='mean', std_f='std').reset_index()
)
cv_flights['flight_cv'] = (
    cv_flights['std_f'] / cv_flights['mean_f'].replace(0, np.nan)
).fillna(0)

def longest_streak(group):
    active = sorted(group[group[FLIGHTS_COL] > 0]['period'].dt.to_period('M').unique())
    if not active: return 0
    max_s = s = 1
    for i in range(1, len(active)):
        if (active[i] - active[i-1]).n == 1:
            s += 1; max_s = max(max_s, s)
        else:
            s = 1
    return max_s

print('Computing streaks (~30s)...')
streak_df = (
    activity.groupby(KEY).apply(longest_streak)
    .reset_index().rename(columns={0: 'longest_active_streak'})
)

last_active = (
    activity[activity[FLIGHTS_COL] > 0]
    .groupby(KEY)['period'].max().reset_index()
    .rename(columns={'period': 'last_flight_date'})
)
last_active['months_since_last_flight'] = (
    (PREDICTION_CUTOFF.year  - last_active['last_flight_date'].dt.year)  * 12 +
    (PREDICTION_CUTOFF.month - last_active['last_flight_date'].dt.month)
).clip(lower=0)

cons_df = (
    cv_flights[[KEY, 'flight_cv']]
    .merge(streak_df, on=KEY, how='outer')
    .merge(last_active[[KEY, 'months_since_last_flight']], on=KEY, how='outer')
    .fillna(0)
)
features = features.merge(cons_df, on=KEY, how='left')
for col in ['flight_cv','longest_active_streak','months_since_last_flight']:
    features[col] = features[col].fillna(0)
    register(col, 'behavioural_consistency', 'Flight regularity and streak')

print(f'After consistency features: {features.shape[1]-1} features')

---
## 4. Feature Family 3 — Econometric Features

### 4a. Fixed-Effects Deviation — Member vs Cohort Baseline
**Logic**: A member flying 2 flights/month in a cohort averaging 5 is underperforming. The same member in a cohort averaging 1 is overperforming. Deviation from expectation is the true signal.

In [ ]:
if CARD_COL in members.columns:
    member_tier_map  = members[[KEY, CARD_COL]].drop_duplicates(KEY)
    act_with_tier    = activity.merge(member_tier_map, on=KEY, how='left')
    cohort_baseline  = (
        act_with_tier.groupby([CARD_COL, YEAR_COL])[FLIGHTS_COL].mean()
        .reset_index().rename(columns={FLIGHTS_COL: 'cohort_avg_flights'})
    )
    act_with_base = act_with_tier.merge(cohort_baseline, on=[CARD_COL, YEAR_COL], how='left')
    act_with_base['fe_deviation'] = (
        act_with_base[FLIGHTS_COL] - act_with_base['cohort_avg_flights'].fillna(0)
    )
    fe_features = (
        act_with_base.groupby(KEY)['fe_deviation']
        .agg(mean_fe_deviation='mean', std_fe_deviation='std')
        .reset_index().fillna(0)
    )
    features = features.merge(fe_features, on=KEY, how='left')
    for col in ['mean_fe_deviation','std_fe_deviation']:
        features[col] = features[col].fillna(0)
        register(col, 'econometric_fixed_effects',
                 'Deviation from tier-year cohort average (fixed effects)')
    print(f'After fixed-effects features: {features.shape[1]-1} features')
else:
    print('Skipping fixed-effects — card tier column not available.')

### 4b. Seasonal Controls

In [ ]:
features = features.merge(sea_vol, on=KEY, how='left')
features['seasonal_volatility'] = features['seasonal_volatility'].fillna(0)
register('seasonal_volatility', 'econometric_seasonal',
         'Avg deviation from personal seasonal baseline')

last_flight_season = (
    activity[activity[FLIGHTS_COL] > 0]
    .groupby(KEY)['period'].max().reset_index()
)
last_flight_season['last_flight_quarter']       = last_flight_season['period'].dt.quarter
last_flight_season['last_flight_month_of_year'] = last_flight_season['period'].dt.month

features = features.merge(
    last_flight_season[[KEY,'last_flight_quarter','last_flight_month_of_year']],
    on=KEY, how='left'
)
features['last_flight_quarter']       = features['last_flight_quarter'].fillna(0)
features['last_flight_month_of_year'] = features['last_flight_month_of_year'].fillna(0)
register('last_flight_quarter', 'econometric_seasonal', 'Quarter in which member last flew')

print(f'After econometric features: {features.shape[1]-1} features')

---
## 5. Feature Family 4 — Demographic & CLV Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

demo_cols = [c for c in ['salary','province','marital_status','education',
                          'gender','enrollment_year'] if c in members.columns]
demo_df = members[[KEY] + ([CLV_COL] if CLV_COL in members.columns else []) + demo_cols].copy()

cat_cols = [c for c in demo_df.select_dtypes(include='object').columns if c != KEY]
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    demo_df[col] = le.fit_transform(demo_df[col].astype(str).fillna('unknown'))
    label_encoders[col] = le
    register(col, 'demographic', f'Label-encoded: {col}')

if CLV_COL in members.columns:
    register(CLV_COL, 'demographic', 'Customer Lifetime Value (existing metric)')

features = features.merge(demo_df, on=KEY, how='left')

if CLV_COL in features.columns:
    features['clv_percentile'] = features[CLV_COL].rank(pct=True)
    register('clv_percentile', 'demographic', 'CLV as percentile rank (0-1)')

print(f'After demographic features: {features.shape[1]-1} features')
for col, le in label_encoders.items():
    print(f'  {col}: {list(le.classes_)}')

---
## 6. Attach Churn Labels & Final Assembly

In [ ]:
features = features.merge(labels, on=KEY, how='inner')

null_counts = features.isnull().sum()
null_cols   = null_counts[null_counts > 0]

if len(null_cols) > 0:
    print(f'Filling {len(null_cols)} columns with remaining nulls:')
    for col in null_cols.index:
        if col == 'churned': continue
        if features[col].dtype == 'object':
            features[col] = features[col].fillna('unknown')
        else:
            features[col] = features[col].fillna(features[col].median())
else:
    print('No nulls remaining.')

print(f'\nFinal: {features.shape[0]:,} members x {features.shape[1]} columns')
print(f'Churn rate: {features["churned"].mean():.1%}')

---
## 7. Feature Importance Preview — Correlation with Churn

In [ ]:
numeric_feats = [c for c in features.select_dtypes(include='number').columns
                 if c not in [KEY, 'churned']]
corr_churn = (
    features[numeric_feats + ['churned']].corr()['churned']
    .drop('churned').abs().sort_values(ascending=False)
)

top_n = 20
top_f = corr_churn.head(top_n)

CHURN_COLOR  = '#C0392B'
RETAIN_COLOR = '#2471A3'

def feat_color(f):
    if any(k in f for k in ['hyperbolic','recency','loss','aversion']): return CHURN_COLOR
    if any(k in f for k in ['fe_','seasonal']): return RETAIN_COLOR
    if any(k in f for k in ['clv','salary','tier']): return '#1A8A4A'
    return '#566573'

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(top_n), top_f.values,
        color=[feat_color(f) for f in top_f.index],
        alpha=0.85, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_f.index, fontsize=9)
ax.set_xlabel('|Pearson correlation with churned|')
ax.set_title(f'Top {top_n} features by univariate correlation with churn')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=CHURN_COLOR,  label='Behavioural (hyperbolic / loss aversion)'),
    Patch(facecolor=RETAIN_COLOR, label='Econometric (fixed effects / seasonal)'),
    Patch(facecolor='#1A8A4A',    label='Demographic / CLV'),
    Patch(facecolor='#566573',    label='Raw activity'),
], loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(FIG / '04_feature_correlation_with_churn.png', bbox_inches='tight')
plt.show()

print('Top 10 features by churn correlation:')
print(corr_churn.head(10).to_string())

---
## 8. Behavioural Feature Distributions: Churners vs Retained

In [ ]:
behav_feats = [f for f in [
    'hyperbolic_flight_score','recency_ratio','loss_aversion_score',
    'redemption_ratio','months_since_last_flight','flight_trajectory'
] if f in features.columns]

if behav_feats:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, feat in enumerate(behav_feats[:6]):
        for cv, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
            sub = features[features['churned']==cv][feat].dropna()
            p1, p99 = sub.quantile(0.01), sub.quantile(0.99)
            axes[i].hist(sub.clip(p1,p99), bins=40, alpha=0.5,
                        color=color, label=label, density=True, edgecolor='white')
        axes[i].set_title(feat.replace('_',' ').title(), fontsize=10)
        axes[i].set_ylabel('Density')
        axes[i].legend(fontsize=8)
    for j in range(len(behav_feats), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Behavioural feature distributions: churners vs retained', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG / '04_behavioural_feature_distributions.png', bbox_inches='tight')
    plt.show()

---
## 9. Save Feature Table & Registry

In [ ]:
features.to_csv(PROC / '04_features.csv', index=False)
print(f'Saved: 04_features.csv  ({features.shape[0]:,} rows x {features.shape[1]} cols)')

registry_df = pd.DataFrame(feature_registry)
registry_df.to_csv(PROC / '04_feature_registry.csv', index=False)
print(f'Saved: 04_feature_registry.csv  ({len(registry_df)} features documented)')

import joblib
joblib.dump(label_encoders, PROC / '04_label_encoders.pkl')
print('Saved: 04_label_encoders.pkl')

print('\n' + '='*55)
print('  FEATURE ENGINEERING COMPLETE')
print('='*55)
family_counts = registry_df.groupby('family').size().sort_values(ascending=False)
for family, count in family_counts.items():
    print(f'  {family:<42}: {count} features')
print(f'  {"TOTAL":<42}: {len(registry_df)} features')
print('='*55)
print('\nNext step -> 05_churn_model.ipynb')